In [5]:
print("Hello World")

Hello World


In [6]:
# IT3011 - Theory & Practices in Statistical Modelling
# Group Name: In_Sighted
# Group Assignment: Recognition for Good Work Increases Employee Engagement
# Dataset: IBM HR Analytics Employee Attrition & Performance (Kaggle)
# =============================================================================
 
# ---- Impoerting Libraries ----

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind, f_oneway, pearsonr


from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
 
import warnings
warnings.filterwarnings("ignore")

In [8]:
# Plotting style 

In [9]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f9f9f9",
    "axes.grid": True,
    "grid.alpha": 0.4,
    "font.size": 11,
})
PALETTE = "Set2"

In [10]:
# SECTION 1 — Load & Inspect Data
# The dataset was downloaded from Kaggle (IBM HR Analytics Attrition dataset).
# It contains 1,470 employee records and 35 variables covering demographics,
# job characteristics, compensation, and satisfaction scores.

In [11]:
df = pd.read_csv("C:\\Users\\Mahesh Masinghe\\Desktop\\TPSM Project\\TPSM_HR_Dataset.csv")

In [12]:
print("=" * 60)
print("SECTION 1 — Dataset Overview")
print("=" * 60)
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nAttrition breakdown:\n{df['Attrition'].value_counts()}")
print(f"\nMissing values: {df.isnull().sum().sum()} (none expected in this dataset)")

SECTION 1 — Dataset Overview
Shape: 1470 rows × 35 columns

Attrition breakdown:
Attrition
No     1233
Yes     237
Name: count, dtype: int64

Missing values: 0 (none expected in this dataset)


In [13]:
# These columns carry no analytical value — they are constants across all rows.
cols_to_drop = ["EmployeeCount", "Over18", "StandardHours", "EmployeeNumber"]
df.drop(columns=cols_to_drop, inplace=True)
 
# Encode the target variable as a binary integer (1 = left, 0 = stayed).
# This is needed for correlation and modelling later.
df["AttritionBinary"] = (df["Attrition"] == "Yes").astype(int)

print(f"\nAttrition rate: {df['AttritionBinary'].mean():.1%}")


Attrition rate: 16.1%


In [ ]:
# SECTION 2 — Descriptive Analytics
# Descriptive analytics answers: "What is actually in the data?"
# We summarise key recognition and engagement variables before running tests.

In [15]:
print("\n" + "=" * 60)
print("SECTION 2 — Descriptive Analytics")
print("=" * 60)
 
recognition_vars = [
    "PerformanceRating", "PercentSalaryHike",
    "YearsSinceLastPromotion", "TrainingTimesLastYear",
    "StockOptionLevel", "JobLevel"
]
 
engagement_vars = [
    "JobSatisfaction", "WorkLifeBalance",
    "JobInvolvement", "EnvironmentSatisfaction",
    "RelationshipSatisfaction"
]
 
print("\n--- Recognition Variables ---")
print(df[recognition_vars].describe().round(2))
 
print("\n--- Engagement Variables ---")
print(df[engagement_vars].describe().round(2))


SECTION 2 — Descriptive Analytics

--- Recognition Variables ---
       PerformanceRating  PercentSalaryHike  YearsSinceLastPromotion  \
count            1470.00            1470.00                  1470.00   
mean                3.15              15.21                     2.19   
std                 0.36               3.66                     3.22   
min                 3.00              11.00                     0.00   
25%                 3.00              12.00                     0.00   
50%                 3.00              14.00                     1.00   
75%                 3.00              18.00                     3.00   
max                 4.00              25.00                    15.00   

       TrainingTimesLastYear  StockOptionLevel  JobLevel  
count                1470.00           1470.00   1470.00  
mean                    2.80              0.79      2.06  
std                     1.29              0.85      1.11  
min                     0.00              0.00   

In [16]:
# ---- 2A: Average satisfaction by performance rating -------------------------
# PerformanceRating = 3 (Excellent) or 4 (Outstanding) in this dataset.
# Checking whether higher-rated employees report better satisfaction.

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Average Engagement Scores by Performance Rating", fontsize=13, fontweight="bold")
 
for ax, var in zip(axes, ["JobSatisfaction", "WorkLifeBalance", "JobInvolvement"]):
    means = df.groupby("PerformanceRating")[var].mean()
    ax.bar(means.index.astype(str), means.values, color=sns.color_palette(PALETTE)[:len(means)])
    ax.set_title(var)
    ax.set_xlabel("Performance Rating")
    ax.set_ylabel("Mean Score")
    ax.set_ylim(0, 4.5)
 
plt.tight_layout()
plt.savefig("fig1_satisfaction_by_rating.png", dpi=150)
plt.close()
print("\nSaved → fig1_satisfaction_by_rating.png")


Saved → fig1_satisfaction_by_rating.png


In [17]:
# ---- 2B: Attrition by department --------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5))
dept_attr = df.groupby("Department")["AttritionBinary"].mean().sort_values(ascending=False) * 100
bars = ax.bar(dept_attr.index, dept_attr.values, color=sns.color_palette(PALETTE)[:len(dept_attr)])
ax.set_title("Attrition Rate by Department (%)", fontweight="bold")
ax.set_ylabel("Attrition Rate (%)")
ax.set_xlabel("Department")
for bar, val in zip(bars, dept_attr.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3, f"{val:.1f}%", ha="center", va="bottom")
plt.tight_layout()
plt.savefig("fig2_attrition_by_dept.png", dpi=150)
plt.close()
print("Saved → fig2_attrition_by_dept.png")

Saved → fig2_attrition_by_dept.png


In [18]:
# ---- 2C: Distribution of salary hike by performance rating ------------------
fig, ax = plt.subplots(figsize=(8, 5))
df.boxplot(column="PercentSalaryHike", by="PerformanceRating", ax=ax, patch_artist=True)
ax.set_title("Salary Hike Distribution by Performance Rating", fontweight="bold")
ax.set_xlabel("Performance Rating")
ax.set_ylabel("% Salary Hike")
plt.suptitle("")
plt.tight_layout()
plt.savefig("fig3_salaryhike_by_rating.png", dpi=150)
plt.close()
print("Saved → fig3_salaryhike_by_rating.png")

Saved → fig3_salaryhike_by_rating.png


In [19]:
# ---- 2D: Cross-tabulation — recognition level vs satisfaction ---------------
# JobSatisfaction and PerformanceRating are both ordinal.
# A cross-tab shows how they co-vary in frequency counts.
crosstab = pd.crosstab(df["PerformanceRating"], df["JobSatisfaction"])
print("\nCross-tabulation — PerformanceRating vs JobSatisfaction")
print(crosstab)


Cross-tabulation — PerformanceRating vs JobSatisfaction
JobSatisfaction      1    2    3    4
PerformanceRating                    
3                  241  237  386  380
4                   48   43   56   79


In [20]:
# ---- 2E: Heatmap of correlations among key variables -----------------------
key_cols = recognition_vars + engagement_vars + ["AttritionBinary"]
corr_matrix = df[key_cols].corr()
 
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, ax=ax, linewidths=0.5
)
ax.set_title("Correlation Heatmap — Recognition & Engagement Variables", fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("fig4_correlation_heatmap.png", dpi=150)
plt.close()
print("Saved → fig4_correlation_heatmap.png")

Saved → fig4_correlation_heatmap.png


In [ ]:

# SECTION 3 — Inferential Analytics
# Inferential analytics answers: "Is the pattern statistically significant?"
# We test H₀ (recognition has no effect) against H₁ (recognition has a
# significant positive effect on engagement).

In [22]:
print("\n" + "=" * 60)
print("SECTION 3 — Inferential Analytics")
print("=" * 60)
 
alpha = 0.05  # significance threshold throughout


SECTION 3 — Inferential Analytics


In [ ]:
#  3A: Pearson Correlation — PerformanceRating vs Engagement vars 
# Pearson r measures the linear relationship between two numeric variables.
# A positive r with p < 0.05 supports H₁.

print("\n--- Pearson Correlations: Recognition → Engagement ---")
print(f"{'Variable':<30} {'r':>8} {'p-value':>12} {'Significant':>12}")
print("-" * 65)
 
rec_vars_numeric = ["PerformanceRating", "PercentSalaryHike", "StockOptionLevel"]
eng_vars_numeric = ["JobSatisfaction", "WorkLifeBalance", "JobInvolvement"]
 
for rv in rec_vars_numeric:
    for ev in eng_vars_numeric:
        r, p = pearsonr(df[rv], df[ev])
        sig = "Yes ✓" if p < alpha else "No"
        print(f"  {rv} → {ev:<22} r={r:+.3f}   p={p:.4f}   {sig}")



--- Pearson Correlations: Recognition → Engagement ---
Variable                              r      p-value  Significant
-----------------------------------------------------------------
  PerformanceRating → JobSatisfaction        r=+0.002   p=0.9299   No
  PerformanceRating → WorkLifeBalance        r=+0.003   p=0.9215   No
  PerformanceRating → JobInvolvement         r=-0.029   p=0.2653   No
  PercentSalaryHike → JobSatisfaction        r=+0.020   p=0.4435   No
  PercentSalaryHike → WorkLifeBalance        r=-0.003   p=0.9000   No
  PercentSalaryHike → JobInvolvement         r=-0.017   p=0.5098   No
  StockOptionLevel → JobSatisfaction        r=+0.011   p=0.6821   No
  StockOptionLevel → WorkLifeBalance        r=+0.004   p=0.8743   No
  StockOptionLevel → JobInvolvement         r=+0.022   p=0.4096   No


In [ ]:
#  3B: Independent T-Test — satisfaction for high vs low PerformanceRating -
# The dataset only has ratings 3 and 4. A t-test checks whether the means
# of satisfaction differ significantly between these two groups.

print("\n--- T-Test: JobSatisfaction — Rating 3 vs Rating 4 ---")
group3 = df[df["PerformanceRating"] == 3]["JobSatisfaction"]
group4 = df[df["PerformanceRating"] == 4]["JobSatisfaction"]
t_stat, p_val = ttest_ind(group3, group4, equal_var=False)  # Welch's t-test
print(f"  Mean (Rating 3): {group3.mean():.3f} | Mean (Rating 4): {group4.mean():.3f}")
print(f"  t = {t_stat:.3f}, p = {p_val:.4f} → {'Reject H₀' if p_val < alpha else 'Fail to reject H₀'}")


--- T-Test: JobSatisfaction — Rating 3 vs Rating 4 ---
  Mean (Rating 3): 2.727 | Mean (Rating 4): 2.735
  t = -0.085, p = 0.9323 → Fail to reject H₀


In [ ]:
#  3C: One-Way ANOVA — satisfaction across Job Levels 
# ANOVA tests whether at least one JobLevel group has a different mean
# satisfaction compared to the others. It extends the t-test to >2 groups.

print("\n--- ANOVA: JobSatisfaction across JobLevels ---")
groups = [grp["JobSatisfaction"].values for _, grp in df.groupby("JobLevel")]
f_stat, p_anova = f_oneway(*groups)
print(f"  F = {f_stat:.3f}, p = {p_anova:.4f} → {'Reject H₀' if p_anova < alpha else 'Fail to reject H₀'}")


--- ANOVA: JobSatisfaction across JobLevels ---
  F = 0.222, p = 0.9265 → Fail to reject H₀


In [ ]:
#  3D: Chi-Square Test — Attrition vs OverTime 
# Chi-square tests independence between two categorical variables.
# Here: does working overtime relate to whether someone leaves?

print("\n--- Chi-Square: Attrition vs OverTime ---")
ct = pd.crosstab(df["Attrition"], df["OverTime"])
chi2, p_chi, dof, expected = chi2_contingency(ct)
print(f"  χ² = {chi2:.3f}, df = {dof}, p = {p_chi:.4f} → {'Reject H₀' if p_chi < alpha else 'Fail to reject H₀'}")


--- Chi-Square: Attrition vs OverTime ---
  χ² = 87.564, df = 1, p = 0.0000 → Reject H₀


In [ ]:
#  3E: Chi-Square — Attrition vs StockOptionLevel 

print("\n--- Chi-Square: Attrition vs StockOptionLevel ---")
ct2 = pd.crosstab(df["Attrition"], df["StockOptionLevel"])
chi2b, p_chi2b, dof2, _ = chi2_contingency(ct2)
print(f"  χ² = {chi2b:.3f}, df = {dof2}, p = {p_chi2b:.4f} → {'Reject H₀' if p_chi2b < alpha else 'Fail to reject H₀'}")


--- Chi-Square: Attrition vs StockOptionLevel ---
  χ² = 60.598, df = 3, p = 0.0000 → Reject H₀


In [ ]:
#  3F: Visualise attrition rate across satisfaction levels 

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Attrition Rate by Satisfaction and Work-Life Balance", fontweight="bold")
 
for ax, var in zip(axes, ["JobSatisfaction", "WorkLifeBalance"]):
    rates = df.groupby(var)["AttritionBinary"].mean() * 100
    ax.bar(rates.index.astype(str), rates.values, color=sns.color_palette("Reds_d", len(rates)))
    ax.set_title(f"Attrition by {var}")
    ax.set_xlabel(var)
    ax.set_ylabel("Attrition Rate (%)")
 
plt.tight_layout()
plt.savefig("fig5_attrition_satisfaction.png", dpi=150)
plt.close()
print("\nSaved → fig5_attrition_satisfaction.png")


Saved → fig5_attrition_satisfaction.png


In [ ]:

# SECTION 4 — Predictive Analytics (Multiple Models)
# We now build and compare several classification models to predict whether
# an employee will leave (Attrition = Yes).
# Models used: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting

In [37]:
print("\n" + "=" * 60)
print("SECTION 4 — Predictive Analytics")
print("=" * 60)


SECTION 4 — Predictive Analytics


In [ ]:
# 4A: Feature Engineering 
# Encode categorical variables so all inputs are numeric.

df_model = df.copy()
label_enc_cols = ["BusinessTravel", "Department", "EducationField",
                  "Gender", "JobRole", "MaritalStatus", "OverTime"]
 
le = LabelEncoder()
for col in label_enc_cols:
    df_model[col] = le.fit_transform(df_model[col])
 
# Features chosen based on our analytical framework.
feature_cols = [
    "PerformanceRating", "PercentSalaryHike", "YearsSinceLastPromotion",
    "TrainingTimesLastYear", "StockOptionLevel", "JobLevel",
    "MonthlyIncome", "JobSatisfaction", "WorkLifeBalance", "JobInvolvement",
    "EnvironmentSatisfaction", "RelationshipSatisfaction",
    "Age", "Department", "YearsAtCompany", "DistanceFromHome",
    "OverTime", "BusinessTravel", "NumCompaniesWorked", "TotalWorkingYears"
]
 
X = df_model[feature_cols]
y = df_model["AttritionBinary"]
 
# Stratified split: preserves the 16% / 84% class ratio in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
# Scale features for Logistic Regression (tree models do not need this).
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
 
print(f"\nTraining set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")
 


Training set: 1176 samples | Test set: 294 samples


In [ ]:
# 4B: Define and Train Models 
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42),
}
 
results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
print("\n--- Cross-Validated ROC-AUC Scores (5-fold) ---")
print(f"{'Model':<25} {'Mean AUC':>10} {'Std':>8}")
print("-" * 45)
 
for name, model in models.items():
    if name == "Logistic Regression":
        Xtr, Xte = X_train_sc, X_test_sc
    else:
        Xtr, Xte = X_train, X_test
 
    cv_scores = cross_val_score(model, Xtr, y_train, cv=skf, scoring="roc_auc")
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
 
    results[name] = {
        "model": model, "y_pred": y_pred, "y_prob": y_prob,
        "auc": auc, "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std(),
        "Xte": Xte
    }
    print(f"  {name:<23} {cv_scores.mean():.4f}   ±{cv_scores.std():.4f}")
